In [21]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from dotenv import load_dotenv
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated, Literal
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

True

In [22]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [23]:
# search tool
search_tool = DuckDuckGoSearchRun(
    name="Internet_search",
    description=(
        "should search updated info from the internet"
        "use this tool when user ask about current events"
        "news, current information, information require"
        "use internet for search"
    )
)

In [24]:
#custom tool
# @tool
# def calculate(first_num: float, second_num: float, operation: str)-> dict:
#     """
#         perform the arithametic operation on the two numbers
#         operations allowed are add, sub, multi, div
#     """
#     try:
#             if operation == "add":
#                 result = first_num + second_num
#             elif operation == "sub":
#                 result = first_num - second_num
#             elif operation == "multi":
#                 result = first_num * second_num
#             elif operation == "div":
#                 if second_num == 0:
#                     return {"error": "ZeroDivisionError please try with number"}
#                 result = first_num/second_num
#             else:
#                 return {"error": "Invalid operation please try with add, sub, multi, div"}
            
#             return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
        
#     except Exception as e:
#             return {"error": str(e)}



In [25]:
#instead of this tool we will add the mcp client
client = MultiServerMCPClient(
    {
        "server-1": {
            "transport": "stdio",
            "command": "python3",
            "args": ["C:\\Users\\Ashutosh Pandey\\Desktop\\Machine Learning\\LangGraph-In-Depth\\main.py"],
        }
    }
)
# now wwe will go to the build graph function

In [26]:
# tool combining 
# tools = [search_tool, calculate]

# # tool binding
# llm_with_tool = llm.bind_tools(tools)

In [27]:
# now lets create a state for chatbot
class chatstate(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [28]:
# we will build a function for building graph
async def build_graph():
    # now we will fetch the tools present in the mcp-server
    tools = await client.get_tools()
    
    print(tools)
    
    # with this tool we will bind the llm
    llm_with_tool = llm.bind_tools(tools)
    
    # lets create out node
    async def chat_node(state: chatstate):
        """LLM node that may answer or request a tool call"""
        messages = state['messages']
        response = await llm_with_tool.ainvoke(messages)  # await and async invoke written as(ainvoke)
        return {'messages': [response]}
    
    # our tool node
    tool_node = ToolNode(tools) # toolnode dont need async since its implementation is async internally.
    
    
    # now lets create our graph
    graph = StateGraph(chatstate)
    
    graph.add_node("chat_node", chat_node)
    graph.add_node("tools", tool_node)
    
    graph.add_edge(START, "chat_node")
    graph.add_conditional_edges("chat_node", tools_condition)
    
    graph.add_edge("tools", "chat_node")
    
    chatbot = graph.compile()
    
    return chatbot

In [31]:
async def main():
    # we will get chatbot from build_graph_function
    chatbot = await build_graph()
    
    # the response also with ainvoke
    response = await chatbot.ainvoke({'messages': [HumanMessage(content="write 10 lines about ancient egypt")]})
    print(response['messages'][-1].content)
    
# if __name__ == "__main__":
#     asyncio.run(main()) # this yu cannot use since this is ipynb file if its a .py file it must have workd
# so use
# await main()